# 07. От DiT к CrossDiT: как denoiser устроен изнутри

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** проследить путь latent→patches→blocks→velocity, отделить time/pooled modulation от token-level text cross-attention и отличить velocity head от π-Flow grid head

**Перед началом:** K6 05–06 (patches, AdaLN, Q/K/V, RoPE); Lecture 4 / Lab 3. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.1–3.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/components/piflow_dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/piflow_dit.py)
- [`core/tensors.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/tensors.py)
- [`pipeline/config.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/pipeline/config.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '07_dit_and_text_crossdit'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 0. Вопрос, вход и выход

**Вопрос лабораторной:** сеть получает шумный latent, время и текст. Что она возвращает,
и через какую ветвь каждое условие влияет на ответ?

Denoiser — функция $u_\theta(x_t, t, y)$, которая возвращает tensor **той же формы, что физический latent**.
У base checkpoint это velocity, у distilled checkpoint — набор grid predictions (раздел 6).
Этот notebook рассматривает video-only T2V ветвь класса `DiffusionTransformer3D`; аудио и обмен между потоками — в 08.

| Объект | Форма (batched, channel-last) | Смысл |
|---|---|---|
| $x_t$ | `[B, T, H, W, 16]` | шумный video latent: B примеров, T latent frames, сетка H×W, 16 каналов VAE |
| visual input | `[B, T, H, W, 33]` | $x_t$ ‖ condition latent ‖ mask = 16+16+1 (`_build_video_input`) |
| text tokens | `[B, L, 3584]` | Qwen2.5-VL hidden states, по вектору на token; `attention_mask [B, L]` отмечает padding |
| pooled | `[B, 768]` | CLIP pooled vector: **один** вектор на prompt |
| time | `[B]` | `1000·t`, где в K6 t=1 — шум, t=0 — данные |
| output | `[B, T, H, W, 16]` | velocity только для физических каналов, без condition/mask |

**Соглашение о времени.** Pinned `flow_match_timesteps` идёт от t=1 к t=0. В конспекте MIT t=0 — шум, t=1 — данные.
Если $t_{MIT}=1-t_{K6}$, то $dx/dt_{MIT}=-dx/dt_{K6}$: тот же путь, противоположный знак velocity.
Вывод target и проверка знака — в [09](09_flow_matching_training.ipynb).


In [ ]:
import re
lite=json.loads((ROOT/'reference/hf/Kandinsky-6.0-Lite-5s-Diffusers/transformer/config.json').read_text())
distill=config('transformer');vae=config('vae')
recipe=(ROOT/'reference/kandinsky-6/kandinsky/configs/checkpoints/lite-distill.yaml').read_text()
gen={k:int(re.search(rf'^\s*{k}:\s*(\d+)',recipe,re.M).group(1)) for k in ['height','width','sample_frames','latent_frames']}
T=(gen['sample_frames']-1)//vae['temporal_compression_ratio']+1
H,W=gen['height']//vae['spatial_compression_ratio'],gen['width']//vae['spatial_compression_ratio']
assert T==gen['latent_frames']
pT,pH,pW=lite['patch_size'];C_in=2*lite['in_visual_dim']+1 if lite['visual_cond'] else lite['in_visual_dim']
head_dim=sum(lite['axes_dims'])
contract={'recipe (pixels)':[gen['sample_frames'],gen['height'],gen['width']],
          'latent [T,H,W,C]':[T,H,W,lite['in_visual_dim']],'visual input channels':C_in,
          'token grid [T/pT,H/pH,W/pW]':[T//pT,H//pH,W//pW],'video tokens N_v':(T//pT)*(H//pH)*(W//pW),
          'patch features':pT*pH*pW*C_in,'model width D / heads':[lite['model_dim'],lite['model_dim']//head_dim],
          'time_dim':lite['time_dim'],'output per token, base velocity':pT*pH*pW*lite['out_visual_dim'],
          'output per token, distill HF config':pT*pH*pW*distill['out_visual_dim']}
assert contract['video tokens N_v']==31*30*54
metrics['lite_contract']=contract
for k,v in contract.items():print(f'{k:36s}',v)


Число tokens определяется сеткой latent и patch size, а не числом каналов:
$N_v=\frac{T}{p_T}\cdot\frac{H}{p_H}\cdot\frac{W}{p_W}$. Каналы и patch определяют только длину вектора до проекции:
$p_Tp_Hp_W C_{in}=4\cdot33=132\to D=1792$. Head width на выходе — $p_Tp_Hp_W C_{out}$.

## 1. Что означает DiT и почему K6 использует два канала conditioning

В [DiT (Peebles & Xie)](https://arxiv.org/abs/2212.09748) transformer обрабатывает patches latent image вместо U-Net.
Авторы сравнили четыре способа подать timestep и class label: in-context tokens, cross-attention block,
adaLN и adaLN-Zero. Для **одного** вектора условия (time + class) лучшим оказался adaLN-Zero.

Текст — последовательность из L векторов. Один pooled vector теряет порядок слов и детали,
поэтому Kandinsky ([K5 report](https://arxiv.org/abs/2511.14993), K6 §3.1) использует оба механизма:

- **global path:** time + CLIP pooled → modulation (shift, scale, gate) каждого sub-layer;
- **token path:** Qwen tokens → text refiner → K/V в cross-attention, Q — visual tokens.

Training target — отдельное решение: DiT предсказывал noise, Kandinsky FM — velocity, distilled K6 — grid policy.
Слово CrossDiT здесь означает DiT с text cross-attention, а не image-classification CrossViT с patches двух размеров.

## 2. Полный forward одной формулой

$$
c = \mathrm{MLP}\big(\mathrm{sincos}(1000t)\big) + \mathrm{LN}(W_p\,\mathrm{pooled}),
\qquad e' = \mathrm{Refiner}_c\big(\mathrm{LN}(W_e\,\mathrm{tokens})\big),
$$

$$
h_0 = W_{in}\,\mathrm{patchify}(x_{in}),\qquad h_{k+1}=\mathrm{Block}_k(h_k;\,e',c),\qquad
u=\mathrm{unpatchify}\big(W_{out}((1+s(c))\,\mathrm{LN}(h_K)+b(c))\big).
$$

Text refiner тоже модулируется $c$: refined tokens зависят от времени. Поэтому кэшируются только
time-independent проекции $W_e$ и $W_p$ ([02](02_text_conditioning.ipynb)), а не выход refiner.
Синусоида строится на ширине `model_dim`, MLP переводит её в `time_dim`; порядок признаков — сначала cos, затем sin.
Проверим это на native классе.


In [ ]:
D=native();tensors=native('core.tensors')
time_encoder=D.TimeEmbeddings(48,16);t=torch.tensor([500.,520.])
with torch.no_grad():
    native_c=time_encoder(t)
    args=torch.outer(t,tensors.get_freqs(24));features=torch.cat([args.cos(),args.sin()],-1)
    emb=time_encoder.timestep_embedder;manual_c=emb.linear_2(emb.act(emb.linear_1(features)))
error=(native_c-manual_c).abs().max().item();assert error<1e-6
metrics['time_embedding_manual_error']=error
dt=np.linspace(-500,500,401)
freqs=tensors.get_freqs(lite['model_dim']//2).double()
def sincos(value):
    a=torch.as_tensor(value,dtype=torch.float64)[...,None]*freqs
    return torch.cat([a.cos(),a.sin()],-1)
similarity=F.cosine_similarity(sincos(500.).expand(len(dt),-1),sincos(500.+torch.tensor(dt)),dim=-1)
fig,ax=plt.subplots(figsize=(7,3));ax.plot(dt,similarity)
ax.set_xlabel('Time offset Δ in DiT units (1000·t), reference t=500')
ax.set_ylabel('Cosine similarity')
ax.set_title(f'Sinusoidal features before MLP, width {lite["model_dim"]}; no learned weights')
fig.tight_layout();fig.savefig(figure_path(LAB,'time_feature_similarity'));plt.show()
metrics['time_features_cos_at_delta_10']=float(similarity[np.argmin(abs(dt-10))])
print('manual time-embedding error:',error,'cos(Δ=10):',metrics['time_features_cos_at_delta_10'])


Признаки различают даже близкие времена (при Δ=10 cosine заметно меньше 1), затем сходство медленно убывает.
Синусоида — фиксированное кодирование без весов; как время меняет обработку, учат MLP и modulation.

## 3. adaLN-Zero: свежая модель не видит ни текст, ни время

Modulation vectors идут в порядке shift, scale, gate ([05](05_nn_patches_and_modulation.ipynb)).
Text refiner block имеет 2 ветви × 3 = 6 векторов, visual block — 3 ветви × 3 = 9, `OutLayer` — 2 (shift, scale, без gate).
Все `Modulation` в K6 инициализируются нулями. Поэтому при инициализации:

- каждый residual gate равен 0, каждый block — тождественное отображение;
- shift=scale=0 в `OutLayer`, значит выход не зависит от $c$;
- выход **не** равен нулю: в отличие от официального кода DiT, где final linear тоже обнулён,
  `OutLayer.out_layer` в K6 получает обычную инициализацию.

Следовательно, свежая сеть — $u=\mathrm{unpatchify}(W_{out}\,\mathrm{LN}(W_{in}\,\mathrm{patchify}(x)))$.
Проверим три предсказания: равенство этой композиции и нулевую чувствительность к text tokens, pooled и времени.


In [ ]:
from labkit.torch_reference import tiny_config
cfg=tiny_config();cfg.update(is_multimodal=False,out_audio_dim=4)
fresh=D.DiffusionTransformer3D(**cfg).eval()
video=torch.randn(1,2,4,4,4)
kw=dict(x_video=torch.cat((video,torch.zeros_like(video),torch.zeros_like(video[...,:1])),-1),
        x_audio=None,text_embed=torch.randn(1,5,12),pooled_text_embed=torch.randn(1,8),
        time=torch.tensor([500.]),visual_rope=fresh.visual_rope((2,2,2),[torch.arange(2)]*3),
        audio_rope=None,text_rope=fresh.text_rope(torch.arange(5)),
        attention_mask=torch.ones(1,5,dtype=torch.bool))
def changed(**update):
    other=dict(kw);other.update(update)
    with torch.no_grad():return (fresh_forward(fresh,kw)-fresh_forward(fresh,other)).abs().max().item()
with torch.no_grad():
    out=fresh_forward(fresh,kw)
    composition=fresh.out_layer(fresh.visual_embeddings(kw['x_video']),torch.zeros(1,cfg['time_dim']))
fresh_rows={'composition error':(out-composition).abs().max().item(),
            'text tokens':changed(text_embed=kw['text_embed']+1),
            'pooled':changed(pooled_text_embed=kw['pooled_text_embed']+1),
            'time':changed(time=torch.tensor([100.])),'output mean |u|':out.abs().mean().item()}
assert fresh_rows['composition error']<1e-6 and fresh_rows['output mean |u|']>0
assert fresh_rows['text tokens']==fresh_rows['pooled']==fresh_rows['time']==0
metrics['fresh_model_invariants']=fresh_rows
print(json.dumps(fresh_rows,indent=2))


Обучение должно открыть gates, иначе condition не влияет на velocity. Нулевой gate не означает нулевой gradient:
производная по весам `Modulation` ненулевая ([05](05_nn_patches_and_modulation.ipynb), раздел 2).

### Обратный путь: unpatchify имеет свой порядок признаков

Вход патча раскладывается как $(p_T,p_H,p_W,C)$, выход `OutLayer` читает признаки как $(C,p_T,p_H,p_W)$.
Если взять единичные $W_{in}$ и $W_{out}$ и убрать LayerNorm, round-trip **не** восстановит latent.
Перестановочная матрица в $W_{out}$ восстанавливает его точно. Обученная сеть выучивает любую такую перестановку,
поэтому parity переноса проверяется по полной функции, а не по предположению о layout.


In [ ]:
C=3;P=(1,2,2);width=math.prod(P)*C
x=torch.arange(1*2*4*4*C,dtype=torch.float32).reshape(1,2,4,4,C)
emb=D.VisualEmbeddings(C,width,P);head=D.OutLayer(width,8,C,P);head.norm=nn.Identity()
order=torch.arange(width).reshape(*P,C).permute(3,0,1,2).flatten()
with torch.no_grad():
    emb.in_layer.weight.copy_(torch.eye(width));emb.in_layer.bias.zero_();head.out_layer.bias.zero_()
    head.out_layer.weight.copy_(torch.eye(width));naive=head(emb(x),torch.zeros(1,8))
    head.out_layer.weight.copy_(torch.eye(width)[order]);permuted=head(emb(x),torch.zeros(1,8))
roundtrip={'identity W_out max error':(naive-x).abs().max().item(),
           'permuted W_out max error':(permuted-x).abs().max().item()}
assert roundtrip['identity W_out max error']>0 and roundtrip['permuted W_out max error']==0
metrics['unpatchify_roundtrip']=roundtrip;print(roundtrip)


## 4. Text CrossDiT: visual query, text K/V

Visual decoder block делает три gated residual updates: self-attention, cross-attention к refined text, FFN.

$$
h \leftarrow h + g_{ca}\odot \mathrm{Attn}\big(Q=\tilde h W_Q,\;K=e'W_K,\;V=e'W_V\big),\qquad
\tilde h=(1+s_{ca})\,\mathrm{LN}(h)+b_{ca}.
$$

Q приходит от video tokens, поэтому выход имеет $N_v$ строк и обновляет video. K/V — от $L$ text tokens;
`attention_mask` закрывает padded keys. Сначала посмотрим формы через hooks на native video-only CrossDiT
(уменьшенные размеры, случайные веса), затем включим малые случайные modulation weights для sensitivity probe.


In [ ]:
model=D.DiffusionTransformer3D(**cfg).eval();activate_gates(model)
shapes={};hooks=[]
def capture(name):
    def hook(module,inputs,output):
        shapes[name]=list(output.shape)
    return hook
for name,mod in [('patches',model.visual_embeddings),('text_projection',model.text_embeddings),
                 ('text_refiner',model.text_blocks[0]),('visual_block',model.visual_transformer_blocks[0]),
                 ('out_layer',model.out_layer)]:
    hooks.append(mod.register_forward_hook(capture(name)))
with torch.no_grad():out=fresh_forward(model,kw)
for h in hooks:h.remove()
assert out.shape==video.shape
metrics['native_shapes']=shapes
print(json.dumps(shapes,indent=2))


Patches выходят как `[B, T', H', W', D]`, внутри blocks они развёрнуты в `[B, N_v, D]`, `OutLayer` возвращает latent shape.

### Какая ветвь переносит какое условие

Опыт 2×2: возмущаем token states **или** pooled vector; text cross-attention работает **или** её выход занулён hook-ом во всех blocks.
Предсказание: token perturbation проходит только через cross-attention; pooled perturbation проходит через modulation и после cut.
Третий контроль: изменяем только padded tokens, закрытые маской, — выход не должен измениться.


In [ ]:
token_kw=dict(kw,text_embed=kw['text_embed']+torch.randn_like(kw['text_embed']))
pooled_kw=dict(kw,pooled_text_embed=kw['pooled_text_embed']+torch.randn_like(kw['pooled_text_embed']))
def delta(other,cut=False):
    handles=[b.cross_attention.register_forward_hook(lambda m,i,o:torch.zeros_like(o))
             for b in model.visual_transformer_blocks] if cut else []
    with torch.no_grad():value=(fresh_forward(model,kw)-fresh_forward(model,other)).abs().max().item()
    for h in handles:h.remove()
    return value
paths={'tokens, CA active':delta(token_kw),'tokens, CA cut':delta(token_kw,True),
       'pooled, CA active':delta(pooled_kw),'pooled, CA cut':delta(pooled_kw,True)}
mask=torch.tensor([[True,True,True,False,False]])
padded=dict(kw,attention_mask=mask);noisy_pad=dict(padded,text_embed=kw['text_embed'].clone())
noisy_pad['text_embed'][:,3:]+=10
with torch.no_grad():
    paths['padded tokens changed']=(fresh_forward(model,padded)-fresh_forward(model,noisy_pad)).abs().max().item()
assert paths['tokens, CA active']>0 and paths['tokens, CA cut']<1e-6
assert paths['pooled, CA cut']>0 and paths['padded tokens changed']<1e-6
metrics['condition_paths']=paths
fig,ax=plt.subplots(figsize=(8,3));ax.barh(list(paths),[max(v,1e-12) for v in paths.values()]);ax.set_xscale('log')
ax.set_xlabel('Max |Δ velocity| (log; values below 1e-12 drawn at 1e-12)')
ax.set_title('Random-weight tiny CrossDiT: which branch carries which condition')
fig.tight_layout();fig.savefig(figure_path(LAB,'condition_paths'));plt.show();print(paths)


Это проверка wiring на случайных весах. Она показывает, **через какую ветвь** возможна зависимость,
но не измеряет, насколько хорошо pretrained модель следует prompt.

## 5. Velocity head и π-Flow grid head

Base checkpoint возвращает на token $p_Tp_Hp_W\cdot16=64$ числа: velocity в каждой ячейке patch.
Distilled Lite возвращает `n_grid` предсказаний на ячейку, из них sampler строит DX policy ([10](10_piflow_distillation.ipynb)).
Архитектура blocks та же; меняется только ширина `OutLayer`/`OutLayerAudio` и интерпретация выхода.

Тонкость config: HF config distill уже хранит **расширенную** ширину (160=16·10, 400=40·10),
а `PiFlowDiffusionTransformer3D` умножает базовую ширину на `n_grid` сам. Поэтому pinned loader
(`_per_grid_overrides`) отбрасывает расширенный override и оставляет базовую ширину module config.
Иначе голова получилась бы в `n_grid` раз шире.


In [ ]:
base_cfg=lite;grid_cfg=distill
n_grid=int(re.search(r'^\s*n_grid:\s*(\d+)',recipe,re.M).group(1))
heads={'base out_visual_dim':base_cfg['out_visual_dim'],'distill out_visual_dim (HF)':grid_cfg['out_visual_dim'],
       'base out_audio_dim':base_cfg['out_audio_dim'],'distill out_audio_dim (HF)':grid_cfg['out_audio_dim'],'n_grid':n_grid}
assert grid_cfg['out_visual_dim']==n_grid*base_cfg['out_visual_dim']
assert grid_cfg['out_audio_dim']==n_grid*base_cfg['out_audio_dim']
show_source('pipeline/config.py','_per_grid_overrides')
grid_model=tiny_model(grid=3).eval();_,_,grid_kw=tiny_inputs(grid_model)
with torch.no_grad():gv,ga=fresh_forward(grid_model,grid_kw)
heads['tiny grid video [B,n_grid,T,H,W,C]']=list(gv.shape);heads['tiny grid audio [B,n_grid,N_a,C]']=list(ga.shape)
assert gv.shape[1]==ga.shape[1]==3
metrics['velocity_vs_grid_head']=heads;print(json.dumps(heads,indent=2))


## 6. Сопоставление архитектур по контрактам

| Архитектура | Обновляемые tokens | Где condition | Корректное сравнение |
|---|---|---|---|
| [DiT](https://arxiv.org/abs/2212.09748) | image latent patches | time + class → adaLN-Zero (лучший из 4 вариантов) | patching, time conditioning, residual gating |
| Kandinsky text CrossDiT ([K5](https://arxiv.org/abs/2511.14993)) | video patches + отдельный text refiner | Qwen tokens → text cross-attention; time+pooled → modulation | native video-only forward выше |
| K6 dual-stream | video и audio tokens разной width | два text refiner + двунаправленная AV fusion | [08](08_dual_stream_audio_video_crossdit.ipynb) |
| [SD3 MMDiT](https://arxiv.org/abs/2403.03206) | text и image tokens с отдельными весами | совместное self-attention по объединённой последовательности | сравнение дизайна, не буквальный K6 код |

Слово «multimodal» само по себе не определяет порядок операций: сверяйте Q, K/V и residual target в коде.


## Самостоятельное исследование

1. Для 960×1728 при тех же 121 frames предскажите $N_v$ и рост логической self-attention matrix; проверьте кодом из раздела 0.
2. Гипотеза: в свежей модели gradient loss по весам `cross_attention.to_query` равен нулю, а по `visual_modulation` — нет. Проверьте и объясните, почему.
3. Перенесите опыт 2×2 на `pooled` при `activate_gates(std=0.01)` и `0.2`: как масштаб gates меняет чувствительность? Отчёт — таблица из 4 чисел на std.
4. Откройте DiT, K5 report и `TransformerDecoderBlock`: отметьте общие primitives, различие conditioning и prediction head.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете по config посчитать tokens и ширины голов, записать forward формулой с $c$ и $e'$,
объяснить, почему свежая K6 сеть не зависит от condition, и назвать ветвь, через которую проходят tokens и pooled.
Архитектура блока не определяет ни solver, ни target. **Следующий шаг:** [08 — два потока и межмодальный обмен](08_dual_stream_audio_video_crossdit.ipynb).

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 06. Attention и RoPE: текст, видео и аудио](06_attention_and_rope.ipynb) · [08. Dual-stream CrossDiT K6: два потока Audio + Video →](08_dual_stream_audio_video_crossdit.ipynb)
